# CBIS-DDSM — MobileNetV2 + SVM (leakage-free pipeline)

Drop-in replacement for the MIAS pipeline, fixing the issues that block publication:
- **Patient-wise split** (uses CBIS official train/test + GroupKFold by `patient_id`) → no leakage.
- **Augmentation applied to TRAIN ONLY**, after the split.
- Uses the **provided cropped ROI patches** (no manual mask math needed).
- Reports **sensitivity / specificity / AUC with 95% bootstrap CI**, not just accuracy.

Set `DATA_ROOT` to wherever you put the CBIS-DDSM CSVs + images in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
import joblib

In [ ]:
# ============================================================
# CONFIG  -- edit these paths to match your Drive layout
# ============================================================
DATA_ROOT  = '/content/drive/MyDrive/CBIS-DDSM'          # folder holding the CSVs + image tree
IMG_ROOT   = DATA_ROOT                                   # prefix the CSV relative paths resolve against
SAVE_DIR   = '/content/drive/MyDrive/VS_CC/models'

# CBIS ships 4 description CSVs. Use mass + calc together, or just one.
TRAIN_CSVS = ['mass_case_description_train_set.csv',
              'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',
              'calc_case_description_test_set.csv']

IMG_SIZE    = 224
BATCH_SIZE  = 16
ROT_ANGLES  = [0, 90, 180, 270]   # train-only augmentation
RANDOM_SEED = 2021

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Load + clean the CBIS description tables
# ============================================================
def load_tables(csv_names):
    frames = []
    for name in csv_names:
        path = os.path.join(DATA_ROOT, name)
        if not os.path.exists(path):
            print('  [skip] missing:', name)
            continue
        frames.append(pd.read_csv(path))
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]
    return df

train_df = load_tables(TRAIN_CSVS)
test_df  = load_tables(TEST_CSVS)
print('Train rows:', len(train_df), '| Test rows:', len(test_df))
print(train_df.columns.tolist())

In [ ]:
# ============================================================
# Label mapping + path resolution
#   pathology: MALIGNANT -> 1 ; BENIGN / BENIGN_WITHOUT_CALLBACK -> 0
#   we use the CROPPED ROI patch column (already lesion-centred)
# ============================================================
def map_label(p):
    return 1 if str(p).strip().upper() == 'MALIGNANT' else 0

def resolve_path(rel):
    """CBIS paths in the CSVs are relative and sometimes messy; normalise."""
    if pd.isna(rel):
        return None
    rel = str(rel).strip().lstrip('/')
    cand = os.path.join(IMG_ROOT, rel)
    return cand

def prep_frame(df):
    df = df.copy()
    # column name varies: 'cropped_image_file_path' is the ROI patch
    patch_col = 'cropped_image_file_path' if 'cropped_image_file_path' in df.columns \
                else 'image_file_path'
    df['label'] = df['pathology'].apply(map_label)
    df['patch_path'] = df[patch_col].apply(resolve_path)
    df['patient_id'] = df['patient_id'].astype(str)
    return df[['patient_id', 'patch_path', 'label']].dropna(subset=['patch_path'])

train_df = prep_frame(train_df)
test_df  = prep_frame(test_df)

# sanity: no patient leaks across the official split
overlap = set(train_df.patient_id) & set(test_df.patient_id)
print('Patient overlap train/test (should be 0):', len(overlap))
print('Train class balance:', np.bincount(train_df.label))
print('Test  class balance:', np.bincount(test_df.label))

In [ ]:
# ============================================================
# Preprocess one patch:  read -> CLAHE -> resize -> 3ch float
# ============================================================
def preprocess_patch(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    clahe = cv2.createCLAHE(2.0, (8, 8))
    img = clahe.apply(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return np.stack([img] * 3, axis=-1).astype(np.float32)

def rotate(img, angle):
    if angle == 0:
        return img
    code = {90: cv2.ROTATE_90_CLOCKWISE,
            180: cv2.ROTATE_180,
            270: cv2.ROTATE_90_COUNTERCLOCKWISE}[angle]
    return cv2.rotate(img, code)

def build_arrays(df, augment=False):
    """Returns X, y, groups. Augmentation (rotations) ONLY when augment=True."""
    X, y, groups = [], [], []
    angles = ROT_ANGLES if augment else [0]
    for _, row in df.iterrows():
        patch = preprocess_patch(row['patch_path'])
        if patch is None:
            continue
        for a in angles:
            X.append(rotate(patch, a))
            y.append(row['label'])
            groups.append(row['patient_id'])
    return np.array(X), np.array(y), np.array(groups)

In [ ]:
# Augment TRAIN only; TEST stays clean (angle 0)
X_train, y_train, g_train = build_arrays(train_df, augment=True)
X_test,  y_test,  g_test  = build_arrays(test_df,  augment=False)

print('Train patches:', X_train.shape, '| classes:', np.bincount(y_train))
print('Test  patches:', X_test.shape,  '| classes:', np.bincount(y_test))

In [ ]:
# ============================================================
# MobileNetV2 feature extractor (frozen ImageNet)
#   For a stronger paper, fine-tune instead of freezing (see note at bottom).
# ============================================================
base = MobileNetV2(weights='imagenet', include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False
feature_extractor = Sequential([base, GlobalAveragePooling2D()])

def extract_features(model, data):
    feats = []
    for i in range(0, len(data), BATCH_SIZE):
        feats.append(model.predict(data[i:i + BATCH_SIZE], verbose=0))
    return np.vstack(feats)

train_feat = extract_features(feature_extractor, X_train)
test_feat  = extract_features(feature_extractor, X_test)
print('Feature dims:', train_feat.shape, test_feat.shape)

In [ ]:
# ============================================================
# Patient-wise cross-validation on the TRAIN set (honest estimate)
# ============================================================
scaler_cv = StandardScaler()
tf_scaled = scaler_cv.fit_transform(train_feat)

gkf = GroupKFold(n_splits=5)
cv_acc, cv_auc = [], []
for fold, (tr, va) in enumerate(gkf.split(tf_scaled, y_train, groups=g_train)):
    clf = SVC(C=50, gamma=0.001, kernel='rbf', probability=True,
              class_weight='balanced', random_state=42)
    clf.fit(tf_scaled[tr], y_train[tr])
    p  = clf.predict(tf_scaled[va])
    pr = clf.predict_proba(tf_scaled[va])[:, 1]
    cv_acc.append(accuracy_score(y_train[va], p))
    cv_auc.append(roc_auc_score(y_train[va], pr))
    print(f'Fold {fold}: acc={cv_acc[-1]:.3f}  auc={cv_auc[-1]:.3f}')

print(f'\nCV accuracy: {np.mean(cv_acc):.3f} +/- {np.std(cv_acc):.3f}')
print(f'CV ROC-AUC : {np.mean(cv_auc):.3f} +/- {np.std(cv_auc):.3f}')

In [ ]:
# ============================================================
# Final model: train on ALL train data, evaluate on official TEST
# ============================================================
scaler = StandardScaler()
train_feat_s = scaler.fit_transform(train_feat)
test_feat_s  = scaler.transform(test_feat)

svm = SVC(C=50, gamma=0.001, kernel='rbf', probability=True,
          class_weight='balanced', random_state=42)
svm.fit(train_feat_s, y_train)

joblib.dump(svm,    os.path.join(SAVE_DIR, 'svm_cbis.pkl'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler_cbis.pkl'))
print('Saved model + scaler to', SAVE_DIR)

In [ ]:
# ============================================================
# Metrics that reviewers actually ask for
# ============================================================
def sens_spec(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0   # recall of malignant
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec

def auc_ci(y_true, y_prob, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return lo, hi

y_pred = svm.predict(test_feat_s)
y_prob = svm.predict_proba(test_feat_s)[:, 1]

sens, spec = sens_spec(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
lo, hi = auc_ci(y_test, y_prob)

print('Accuracy   :', round(accuracy_score(y_test, y_pred), 4))
print('Sensitivity:', round(sens, 4))
print('Specificity:', round(spec, 4))
print(f'ROC-AUC    : {auc:.4f}  (95% CI {lo:.3f}-{hi:.3f})')
print()
print(classification_report(y_test, y_pred, digits=3))

## External validation (INbreast / CMMD / DMID)

Load the saved SVM + scaler and run on a *different* dataset to report cross-dataset generalization. Provide a CSV with an image-path column, a binary label column, and (optionally) x/y/r for ROI cropping. Edit the column names + paths below.

In [ ]:
# ---- external dataset config ----
EXT_CSV       = '/content/drive/MyDrive/dmid_tiff/df3.csv'   # change me
EXT_PATH_COL  = 'Image_Path'
EXT_LABEL_COL = 'class'
EXT_HAS_ROI   = True            # set False to center-crop every image
EXT_ROI_COLS  = ('x', 'y', 'z') # (cx, cy, radius) column names if EXT_HAS_ROI
FALLBACK_CROP = 600             # px box used when ROI is missing/invalid

ext_df = pd.read_csv(EXT_CSV, encoding='ISO-8859-1')
print('External samples:', len(ext_df), '| classes:', np.bincount(ext_df[EXT_LABEL_COL].astype(int)))

In [ ]:
def center_crop(img, size):
    h, w = img.shape[:2]
    cx, cy, half = w // 2, h // 2, size // 2
    return img[max(0, cy - half):cy + half, max(0, cx - half):cx + half]

def crop_roi(img, x, y, r):
    x, y, r = int(float(x)), int(float(y)), int(float(r))
    h, w = img.shape[:2]
    return img[max(0, y - r):min(h, y + r), max(0, x - r):min(w, x + r)]

def preprocess_from_gray(gray):
    clahe = cv2.createCLAHE(2.0, (8, 8))
    gray = clahe.apply(gray)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    return np.stack([gray / 255.0] * 3, axis=-1).astype(np.float32)

# rebuild a frozen extractor so this cell is runnable standalone
base_ext = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_ext.trainable = False
ext_extractor = Sequential([base_ext, GlobalAveragePooling2D()])

svm_ext    = joblib.load(os.path.join(SAVE_DIR, 'svm_cbis.pkl'))
scaler_ext = joblib.load(os.path.join(SAVE_DIR, 'scaler_cbis.pkl'))

Xe, ye = [], []
for _, row in ext_df.iterrows():
    img = cv2.imread(str(row[EXT_PATH_COL]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    roi = None
    if EXT_HAS_ROI:
        try:
            cx, cy, r = (float(row[c]) for c in EXT_ROI_COLS)
            if not any(np.isnan([cx, cy, r])):
                roi = crop_roi(img, cx, cy, r)
        except Exception:
            roi = None
    if roi is None or roi.size == 0:
        roi = center_crop(img, FALLBACK_CROP)
    Xe.append(preprocess_from_gray(roi))
    ye.append(int(row[EXT_LABEL_COL]))

Xe = np.array(Xe); ye = np.array(ye)
feat_e = scaler_ext.transform(extract_features(ext_extractor, Xe))
pred_e = svm_ext.predict(feat_e)
prob_e = svm_ext.predict_proba(feat_e)[:, 1]

sens, spec = sens_spec(ye, pred_e)
lo, hi = auc_ci(ye, prob_e)
print('=== EXTERNAL VALIDATION ===')
print('Accuracy   :', round(accuracy_score(ye, pred_e), 4))
print('Sensitivity:', round(sens, 4))
print('Specificity:', round(spec, 4))
print(f'ROC-AUC    : {roc_auc_score(ye, prob_e):.4f}  (95% CI {lo:.3f}-{hi:.3f})')
print()
print(classification_report(ye, pred_e, digits=3))

## Fine-tuning variant (recommended for the paper)

Frozen ImageNet features under-use a real dataset. Here we unfreeze the top blocks of MobileNetV2 and train end-to-end with a small head. Reuses `X_train/y_train/X_test/y_test` from above. Two options: (A) use the fine-tuned CNN directly as the classifier, or (B) chop the head off and feed the better features back into the SVM.

In [ ]:
from tensorflow.keras.layers import Input, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

ft_base = MobileNetV2(weights='imagenet', include_top=False,
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
ft_base.trainable = True
# freeze everything except the last ~30 layers
for layer in ft_base.layers[:-30]:
    layer.trainable = False

inp = Input((IMG_SIZE, IMG_SIZE, 3))
x = ft_base(inp, training=True)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(1, activation='sigmoid')(x)
ft_model = Model(inp, out)

ft_model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy',
                 metrics=['accuracy'])

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
ft_model.fit(X_train, y_train,
             validation_data=(X_test, y_test),
             epochs=30, batch_size=BATCH_SIZE,
             class_weight=class_weight,
             callbacks=[es], verbose=1)

In [ ]:
# Option A: evaluate the fine-tuned CNN directly
prob_ft = ft_model.predict(X_test, verbose=0).ravel()
pred_ft = (prob_ft >= 0.5).astype(int)

sens, spec = sens_spec(y_test, pred_ft)
lo, hi = auc_ci(y_test, prob_ft)
print('=== FINE-TUNED CNN (option A) ===')
print('Accuracy   :', round(accuracy_score(y_test, pred_ft), 4))
print('Sensitivity:', round(sens, 4))
print('Specificity:', round(spec, 4))
print(f'ROC-AUC    : {roc_auc_score(y_test, prob_ft):.4f}  (95% CI {lo:.3f}-{hi:.3f})')
print()
print(classification_report(y_test, pred_ft, digits=3))

ft_model.save(os.path.join(SAVE_DIR, 'mobilenet_finetuned.keras'))

In [ ]:
# Option B: use fine-tuned backbone as a better feature extractor for the SVM
ft_extractor = Model(ft_model.input,
                     ft_model.get_layer(index=2).output)  # GAP output

ftr_train = ft_extractor.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
ftr_test  = ft_extractor.predict(X_test,  batch_size=BATCH_SIZE, verbose=0)

sc_ft = StandardScaler()
ftr_train_s = sc_ft.fit_transform(ftr_train)
ftr_test_s  = sc_ft.transform(ftr_test)

svm_ft = SVC(C=50, gamma=0.001, kernel='rbf', probability=True,
             class_weight='balanced', random_state=42)
svm_ft.fit(ftr_train_s, y_train)

pred_b = svm_ft.predict(ftr_test_s)
prob_b = svm_ft.predict_proba(ftr_test_s)[:, 1]
sens, spec = sens_spec(y_test, pred_b)
lo, hi = auc_ci(y_test, prob_b)
print('=== FINE-TUNED FEATURES + SVM (option B) ===')
print('Accuracy   :', round(accuracy_score(y_test, pred_b), 4))
print('Sensitivity:', round(sens, 4))
print('Specificity:', round(spec, 4))
print(f'ROC-AUC    : {roc_auc_score(y_test, prob_b):.4f}  (95% CI {lo:.3f}-{hi:.3f})')
print()
print(classification_report(y_test, pred_b, digits=3))

## Results comparison table (paste into the paper)

Collects every model you ran above into one table: accuracy, sensitivity, specificity, AUC (95% CI) and macro-F1. Rows appear only for the cells you actually executed. Saved to `results_comparison.csv`.

In [ ]:
from sklearn.metrics import f1_score

def metric_row(name, y_true, y_pred, y_prob):
    sens, spec = sens_spec(y_true, y_pred)
    lo, hi = auc_ci(y_true, y_prob)
    return {
        'Model':       name,
        'Accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'Sensitivity': round(sens, 4),
        'Specificity': round(spec, 4),
        'AUC':         round(roc_auc_score(y_true, y_prob), 4),
        'AUC_95CI':    f'{lo:.3f}-{hi:.3f}',
        'Macro_F1':    round(f1_score(y_true, y_pred, average='macro'), 4),
    }

rows = []
# Frozen MobileNetV2 + SVM (official CBIS test)
try:
    rows.append(metric_row('Frozen MobileNetV2 + SVM', y_test, y_pred, y_prob))
except NameError:
    pass
# Fine-tuned CNN (option A)
try:
    rows.append(metric_row('Fine-tuned MobileNetV2 (CNN)', y_test, pred_ft, prob_ft))
except NameError:
    pass
# Fine-tuned features + SVM (option B)
try:
    rows.append(metric_row('Fine-tuned features + SVM', y_test, pred_b, prob_b))
except NameError:
    pass
# External validation (optional)
try:
    rows.append(metric_row('Frozen + SVM (EXTERNAL)', ye, pred_e, prob_e))
except NameError:
    pass

results = pd.DataFrame(rows)
results.to_csv(os.path.join(SAVE_DIR, 'results_comparison.csv'), index=False)
print(results.to_string(index=False))

## ROC curves (paper figure)

Plots every model you ran on one axis with AUC in the legend, and saves a 300-dpi `roc_curves.png` ready for the manuscript. Curves appear only for the cells you executed.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# (label, y_true, y_prob) for whichever models exist in memory
curves = []
for name, yt, yp in [
    ('Frozen MobileNetV2 + SVM',     'y_test', 'y_prob'),
    ('Fine-tuned MobileNetV2 (CNN)', 'y_test', 'prob_ft'),
    ('Fine-tuned features + SVM',    'y_test', 'prob_b'),
    ('Frozen + SVM (EXTERNAL)',      'ye',     'prob_e'),
]:
    try:
        curves.append((name, np.asarray(eval(yt)), np.asarray(eval(yp))))
    except NameError:
        continue

plt.figure(figsize=(6, 6))
for name, yt, yp in curves:
    fpr, tpr, _ = roc_curve(yt, yp)
    auc = roc_auc_score(yt, yp)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC — CBIS-DDSM benign vs malignant')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'roc_curves.png'), dpi=300)
plt.show()
print('Saved figure to', os.path.join(SAVE_DIR, 'roc_curves.png'))

## Notes for the paper

1. **External validation.** After this runs, point the same `preprocess_patch` + saved `svm`/`scaler` at **INbreast** and/or **CMMD** to report cross-dataset numbers. That is the section your MIAS draft was missing.
2. **Fine-tuning beats frozen features.** On a dataset this size, unfreeze the top blocks of MobileNetV2 and train end-to-end (or fine-tune then extract). Frozen ImageNet features only looked fine on tiny leaky MIAS.
3. **Report sensitivity/specificity + AUC CI**, never accuracy alone — the classes are imbalanced.
4. Keep `class_weight='balanced'` (replaces the arbitrary 1:5 weight) unless you justify a cost ratio clinically.